In [2]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [3]:
API_KEY = "b01a5d2a-72be-4da7-a0a1-28807d97a2d0"
HOSTNAME = "https://api.cassanova.com"
FECHA_DESDE = "2026-06-01"
FECHA_HASTA = "2026-06-30"
ID_SALES_POINT = None 

In [4]:
def obtener_token(api_key: str) -> str:
    url = f"{HOSTNAME}/apikey/token"
    headers = {
        "Content-Type": "application/json",
        "X-Requested-With": "*",
    }
    resp = requests.post(url, headers=headers, json={"apiKey": api_key})
    resp.raise_for_status()
    data = resp.json()
    return data["access_token"]

token = obtener_token(API_KEY)
print("Token obtenido correctamente." if token else "No se pudo obtener el token.")

Token obtenido correctamente.


In [4]:
# ACCESS_TOKEN = token

# headers = {
#     "Authorization": f"Bearer {ACCESS_TOKEN}",
#     "X-Version": "1.0.0",
#     "Content-Type": "application/json"
# }


# # -----------------------------
# # FUNCIÓN
# # -----------------------------

# def get_receipts(datetime_from, datetime_to, limit=100):

#     url = "https://api.cassanova.com/documents/receipts"

#     all_receipts = []

#     current = datetime.strptime(datetime_from, "%Y-%m-%d")
#     end = datetime.strptime(datetime_to, "%Y-%m-%d")

#     while current <= end:

#         # Intervalo máximo de 2 días
#         chunk_end = min(current + timedelta(days=2), end)

#         print(
#             f"Descargando desde {current.date()} "
#             f"hasta {chunk_end.date()}..."
#         )

#         start = 0

#         while True:

#             params = {
#                 "start": start,
#                 "limit": limit,
#                 "datetimeFrom": current.strftime("%Y-%m-%d"),
#                 "datetimeTo": chunk_end.strftime("%Y-%m-%d"),
#                 # "calculatedAmount": "true"
#             }

#             response = requests.get(
#                 url,
#                 headers=headers,
#                 params=params
#             )

#             # print(response.status_code)
#             if response.status_code != 200:
#                 print(f"FAIL. {response.status_code}: {response.text}")

#             response.raise_for_status()

#             data = response.json()

#             receipts = data.get("receipts", [])

#             if len(receipts) == 0:
#                 break

#             all_receipts.extend(receipts)

#             if len(receipts) < limit:
#                 break

#             start += limit

#         current = chunk_end + timedelta(days=1)


#     return pd.json_normalize(all_receipts)

In [5]:
import time
from datetime import datetime, timedelta

ACCESS_TOKEN = token

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "X-Version": "1.0.0",
    "Content-Type": "application/json"
}

# -----------------------------
# CONTROL DE RATE LIMIT
# -----------------------------
MAX_CALLS = 360
WINDOW_SECONDS = 10 * 60  # 10 minutos

call_count = 0
window_start = time.monotonic()


def registrar_llamada():
    """
    Lleva la cuenta de llamadas dentro de la ventana de 10 minutos.
    Si se llega al límite, espera hasta que la ventana se reinicie.
    """
    global call_count, window_start

    now = time.monotonic()
    elapsed = now - window_start

    # Si ya pasó la ventana, la reiniciamos
    if elapsed >= WINDOW_SECONDS:
        call_count = 0
        window_start = now
        elapsed = 0

    # Si llegamos al límite, esperamos lo que falte de ventana
    if call_count >= MAX_CALLS:
        wait_time = WINDOW_SECONDS - elapsed
        if wait_time > 0:
            print(
                f"Límite de {MAX_CALLS} llamadas alcanzado. "
                f"Esperando {wait_time:.1f} segundos a que se reinicie la ventana..."
            )
            time.sleep(wait_time)

        # Reiniciamos el contador y la ventana tras la espera
        call_count = 0
        window_start = time.monotonic()

    call_count += 1


# -----------------------------
# FUNCIÓN
# -----------------------------

def get_receipts(datetime_from, datetime_to, limit=100):

    url = "https://api.cassanova.com/documents/receipts"

    all_receipts = []

    current = datetime.strptime(datetime_from, "%Y-%m-%d")
    end = datetime.strptime(datetime_to, "%Y-%m-%d")

    while current <= end:

        # Intervalo máximo de 2 días
        chunk_end = min(current + timedelta(days=2), end)

        print(
            f"Descargando desde {current.date()} "
            f"hasta {chunk_end.date()}..."
        )

        start = 0

        while True:

            params = {
                "start": start,
                "limit": limit,
                "datetimeFrom": current.strftime("%Y-%m-%d"),
                "datetimeTo": chunk_end.strftime("%Y-%m-%d"),
                # "calculatedAmount": "true"
            }

            # Controlamos el rate limit antes de cada llamada
            registrar_llamada()

            response = requests.get(
                url,
                headers=headers,
                params=params
            )

            # Si la API devuelve un 429 (rate limit excedido), esperamos
            # y reintentamos la misma petición sin perder datos
            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")
                wait_time = float(retry_after) if retry_after else WINDOW_SECONDS
                print(
                    f"HTTP 429: límite de llamadas excedido según la API. "
                    f"Esperando {wait_time:.1f} segundos antes de reintentar..."
                )
                time.sleep(wait_time)
                # Reiniciamos ventana local también, por seguridad
                global call_count, window_start
                call_count = 0
                window_start = time.monotonic()
                continue  # reintenta la misma petición

            if response.status_code != 200:
                print(f"FAIL. {response.status_code}: {response.text}")

            response.raise_for_status()

            data = response.json()

            receipts = data.get("receipts", [])

            if len(receipts) == 0:
                break

            all_receipts.extend(receipts)

            if len(receipts) < limit:
                break

            start += limit

        current = chunk_end + timedelta(days=1)

    df = pd.json_normalize(all_receipts)

    df = df.rename(columns={
        "document.idSalesPoint": "idSalesPoint",
        "document.amount": "amount",
        "document.datetime": "datetime",
    })

    df["datetime"] = pd.to_datetime(df["datetime"])

    df["datetime"] = df["datetime"].dt.floor("h")

    ventas_hora = (
        df.groupby("datetime", as_index=False)
        .agg(
            ventas=("amount", "sum")
        )
        .sort_values("datetime")
    )
    
    return ventas_hora

In [6]:
df_24 = get_receipts("2024-01-01", "2024-12-31")

Descargando desde 2024-01-01 hasta 2024-01-03...
Descargando desde 2024-01-04 hasta 2024-01-06...
Descargando desde 2024-01-07 hasta 2024-01-09...
Descargando desde 2024-01-10 hasta 2024-01-12...
Descargando desde 2024-01-13 hasta 2024-01-15...
Descargando desde 2024-01-16 hasta 2024-01-18...
Descargando desde 2024-01-19 hasta 2024-01-21...
Descargando desde 2024-01-22 hasta 2024-01-24...
Descargando desde 2024-01-25 hasta 2024-01-27...
Descargando desde 2024-01-28 hasta 2024-01-30...
Descargando desde 2024-01-31 hasta 2024-02-02...
Descargando desde 2024-02-03 hasta 2024-02-05...
Descargando desde 2024-02-06 hasta 2024-02-08...
Descargando desde 2024-02-09 hasta 2024-02-11...
Descargando desde 2024-02-12 hasta 2024-02-14...
Descargando desde 2024-02-15 hasta 2024-02-17...
Descargando desde 2024-02-18 hasta 2024-02-20...
Descargando desde 2024-02-21 hasta 2024-02-23...
Descargando desde 2024-02-24 hasta 2024-02-26...
Descargando desde 2024-02-27 hasta 2024-02-29...
Descargando desde 20

In [7]:
df_24.to_csv("data/sales/sales_per_hour_2024.csv", index=False)

In [6]:
df_25 = get_receipts("2025-01-01", "2025-12-31")
df_25.to_csv("data/sales/sales_per_hour_2025.csv", index=False)

Descargando desde 2025-01-01 hasta 2025-01-03...
Descargando desde 2025-01-04 hasta 2025-01-06...
Descargando desde 2025-01-07 hasta 2025-01-09...
Descargando desde 2025-01-10 hasta 2025-01-12...
Descargando desde 2025-01-13 hasta 2025-01-15...
Descargando desde 2025-01-16 hasta 2025-01-18...
Descargando desde 2025-01-19 hasta 2025-01-21...
Descargando desde 2025-01-22 hasta 2025-01-24...
Descargando desde 2025-01-25 hasta 2025-01-27...
Descargando desde 2025-01-28 hasta 2025-01-30...
Descargando desde 2025-01-31 hasta 2025-02-02...
Descargando desde 2025-02-03 hasta 2025-02-05...
Descargando desde 2025-02-06 hasta 2025-02-08...
Descargando desde 2025-02-09 hasta 2025-02-11...
Descargando desde 2025-02-12 hasta 2025-02-14...
Descargando desde 2025-02-15 hasta 2025-02-17...
Descargando desde 2025-02-18 hasta 2025-02-20...
Descargando desde 2025-02-21 hasta 2025-02-23...
Descargando desde 2025-02-24 hasta 2025-02-26...
Descargando desde 2025-02-27 hasta 2025-03-01...
Descargando desde 20

In [7]:
df_26 = get_receipts("2026-01-01", "2026-06-30")
df_26.to_csv("data/sales/sales_per_hour_2026.csv", index=False)

Descargando desde 2026-01-01 hasta 2026-01-03...
Descargando desde 2026-01-04 hasta 2026-01-06...
Descargando desde 2026-01-07 hasta 2026-01-09...
Descargando desde 2026-01-10 hasta 2026-01-12...
Descargando desde 2026-01-13 hasta 2026-01-15...
Descargando desde 2026-01-16 hasta 2026-01-18...
Descargando desde 2026-01-19 hasta 2026-01-21...
Descargando desde 2026-01-22 hasta 2026-01-24...
Descargando desde 2026-01-25 hasta 2026-01-27...
Descargando desde 2026-01-28 hasta 2026-01-30...
Descargando desde 2026-01-31 hasta 2026-02-02...
Descargando desde 2026-02-03 hasta 2026-02-05...
Descargando desde 2026-02-06 hasta 2026-02-08...
Descargando desde 2026-02-09 hasta 2026-02-11...
Descargando desde 2026-02-12 hasta 2026-02-14...
Descargando desde 2026-02-15 hasta 2026-02-17...
Descargando desde 2026-02-18 hasta 2026-02-20...
Descargando desde 2026-02-21 hasta 2026-02-23...
Descargando desde 2026-02-24 hasta 2026-02-26...
Descargando desde 2026-02-27 hasta 2026-03-01...
Descargando desde 20

Pruebas de comprobación

In [8]:
# df_per_days = (
#     df.groupby(df["datetime"].dt.date, as_index=False)
#       .agg(
#           ventas=("amount", "sum")
#       )
#       .sort_values("datetime")
# )
# df_per_days

In [9]:
# df["datetime"] = pd.to_datetime(df["datetime"])

# # Fecha de negocio
# df["business_date"] = df["datetime"].dt.normalize()

# # Las ventas entre las 00:00 y las 02:59 pertenecen al día anterior
# mask = df["datetime"].dt.hour < 2
# df.loc[mask, "business_date"] -= pd.Timedelta(days=1)

# ventas_dia = (
#     df.groupby("business_date", as_index=False)
#       .agg(
#           ventas=("amount", "sum"),
#           tickets=("id", "count")
#       )
#       .rename(columns={"business_date": "fecha"})
# )

# print(ventas_dia)

In [10]:
# import json
# r = requests.get(
#     "https://api.cassanova.com/documents/receipts",
#     headers=headers,
#     params={"start": 0, "limit": 1, "datetimeFrom": "2024-01-01", "datetimeTo": "2024-01-01"}
# )
# print(json.dumps(r.json()["receipts"][0], indent=2, default=str))

In [11]:
# import requests
# import pandas as pd
# from datetime import datetime, timedelta

# ACCESS_TOKEN = token

# headers = {
#     "Authorization": f"Bearer {ACCESS_TOKEN}",
#     "X-Version": "1.0.0",
#     "Content-Type": "application/json"
# }


# # -----------------------------
# # DESCARGA DE RECEIPTS (tu función original)
# # -----------------------------

# def get_receipts(datetime_from, datetime_to, limit=100):

#     url = "https://api.cassanova.com/documents/receipts"

#     all_receipts = []

#     current = datetime.strptime(datetime_from, "%Y-%m-%d")
#     end = datetime.strptime(datetime_to, "%Y-%m-%d")

#     while current <= end:

#         chunk_end = min(current + timedelta(days=2), end)

#         print(f"\nDescargando desde {current.date()} hasta {chunk_end.date()}...")

#         start = 0

#         while True:

#             params = {
#                 "start": start,
#                 "limit": limit,
#                 "datetimeFrom": current.strftime("%Y-%m-%d"),
#                 "datetimeTo": chunk_end.strftime("%Y-%m-%d"),
#                 "calculatedAmount": "true"
#             }

#             response = requests.get(url, headers=headers, params=params)
#             response.raise_for_status()
#             data = response.json()

#             receipts = data.get("receipts", [])

#             if len(receipts) == 0:
#                 break

#             all_receipts.extend(receipts)

#             print(f"  Página {start//limit + 1}: {len(receipts)} tickets")

#             if len(receipts) < limit:
#                 break

#             start += limit

#         current = chunk_end + timedelta(days=1)

#     print(f"\nTotal tickets descargados: {len(all_receipts)}")

#     return pd.json_normalize(all_receipts)


# # -----------------------------
# # AGRUPACIÓN POR HORA
# # -----------------------------

# def get_sales_by_hour(datetime_from, datetime_to, only_confirmed=False):

#     df = get_receipts(datetime_from, datetime_to)

#     if df.empty:
#         print("No se han descargado tickets.")
#         return df

#     # Filtrar solo confirmados si se pide (evita contar tickets no cerrados)
#     if only_confirmed and "document.confirmed" in df.columns:
#         antes = len(df)
#         df = df[df["document.confirmed"] == True]
#         print(f"Filtrados no confirmados: {antes - len(df)} tickets excluidos")

#     # Parsear la fecha/hora
#     df["datetime"] = pd.to_datetime(df["datetime"])
#     df["date"] = df["datetime"].dt.date
#     df["hour"] = df["datetime"].dt.hour

#     # Agrupar por día y hora
#     resumen = (
#         df.groupby(["date", "hour"])
#         .agg(
#             total_amount=("document.amount", "sum"),
#             num_tickets=("id", "count")
#         )
#         .reset_index()
#         .sort_values(["date", "hour"])
#     )

#     return resumen, df


# if __name__ == "__main__":
#     resumen, df_raw = get_sales_by_hour("2026-06-21", "2026-06-29")
#     print(resumen)

#     # Total por día, para comparar con el dashboard
#     print("\nTotal por día:")
#     print(df_raw.groupby("date")["document.amount"].sum())

In [12]:
# import requests

# def get_sold_by_tax(datetime_from, datetime_to, ids_sales_point=None, limit=100):
#     url = "https://api.cassanova.com/reports/sold/taxes"
#     all_sold = []
#     start = 0
#     data = None

#     while True:
#         params = {
#             "start": start,
#             "limit": limit,
#             "datetimeFrom": f'"{datetime_from}"',   # <-- comillas incluidas
#             "datetimeTo": f'"{datetime_to}"',       # <-- comillas incluidas
#         }
#         if ids_sales_point:
#             params["idsSalesPoint"] = ids_sales_point

#         response = requests.get(url, headers=headers, params=params)
#         response.raise_for_status()
#         data = response.json()

#         all_sold.extend(data.get("sold", []))

#         print(f"totalSold: {data['totalSold']} | totalRefund: {data['totalRefund']} "
#               f"| totalQuantity: {data['totalQuantity']}")

#         if start + limit >= data["totalCount"]:
#             break
#         start += limit

#     return data["totalSold"], data["totalRefund"], all_sold


# total_sold, total_refund, detalle = get_sold_by_tax("2026-06-22", "2026-06-22")
# print("Neto:", float(total_sold) - float(total_refund))

In [13]:
# df = get_receipts("2026-06-22", "2026-06-22")

# print("Nº de tickets:", len(df))
# print("Suma document.amount (todos):", df["document.amount"].sum())

# if "document.confirmed" in df.columns:
#     print("Nº confirmados:", (df["document.confirmed"] == True).sum())
#     print("Nº NO confirmados:", (df["document.confirmed"] == False).sum())
#     print("Suma solo confirmados:", df.loc[df["document.confirmed"] == True, "document.amount"].sum())
#     print("Suma solo NO confirmados:", df.loc[df["document.confirmed"] == False, "document.amount"].sum())

# if "document.taxFree" in df.columns:
#     print("Nº taxFree=True:", (df["document.taxFree"] == True).sum())

In [14]:
# total_sold, total_refund, detalle = get_sold_by_tax("2026-06-22", "2026-06-22")
# print("totalSold:", total_sold)
# print("totalRefund:", total_refund)
# print("Neto:", float(total_sold) - float(total_refund))

In [15]:
# # ¿Hay tickets duplicados por id? (podría pasar por solape entre páginas)
# print("IDs únicos:", df["id"].nunique(), "de", len(df))

# # ¿Algún importe negativo (podría indicar anulaciones/devoluciones mezcladas)?
# print("Tickets con amount negativo:", (df["document.amount"] < 0).sum())
# print(df.loc[df["document.amount"] < 0, ["id", "document.amount"]])

# # Si existe un campo de tipo de documento o estado, míralo
# for col in df.columns:
#     if "type" in col.lower() or "status" in col.lower() or "void" in col.lower() or "refund" in col.lower():
#         print(col, "->", df[col].unique())

In [16]:
# total_sold, total_refund, detalle = get_sold_by_tax("2026-06-22", "2026-06-22")
# print("totalSold:", total_sold)
# print("totalRefund:", total_refund)
# print("Neto:", float(total_sold) - float(total_refund))
# print("Diferencia vs receipts (2239.2):", float(total_sold) - float(total_refund) - 2239.2)

In [17]:
# import requests

# # pedimos los receipts en crudo (sin json_normalize) para acceder a "rows"
# params = {"start": 0, "limit": 100, "datetimeFrom": "2026-06-22", "datetimeTo": "2026-06-22", "calculatedAmount": "true"}
# r = requests.get("https://api.cassanova.com/documents/receipts", headers=headers, params=params)
# receipts_raw = r.json()["receipts"]

# diffs = []
# for rec in receipts_raw:
#     doc = rec["document"]
#     header_amount = doc["amount"]
#     rows_sum = sum(row.get("amount", row.get("price", 0)) for row in doc.get("rows", []))
#     if abs(header_amount - rows_sum) > 0.01:
#         diffs.append((rec["id"], header_amount, rows_sum))

# print(f"Tickets con diferencia cabecera vs líneas: {len(diffs)}")
# for d in diffs[:10]:
#     print(d)

In [18]:
# df_prev = get_receipts("2026-06-21", "2026-06-21")
# df_next = get_receipts("2026-06-23", "2026-06-23")

# df["datetime"] = pd.to_datetime(df["datetime"])
# df_prev["datetime"] = pd.to_datetime(df_prev["datetime"])
# df_next["datetime"] = pd.to_datetime(df_next["datetime"])

# print("Tickets del 22 antes de la 01:00:", (df["datetime"].dt.hour < 1).sum())
# print("Tickets del 22 después de las 23:00:", (df["datetime"].dt.hour >= 23).sum())
# print("Tickets del 21 después de las 23:00:", (df_prev["datetime"].dt.hour >= 23).sum())
# print("Tickets del 23 antes de la 01:00:", (df_next["datetime"].dt.hour < 1).sum())

# print("Suma 21 después de las 23:00:", df_prev.loc[df_prev["datetime"].dt.hour >= 23, "document.amount"].sum())
# print("Suma 23 antes de la 01:00:", df_next.loc[df_next["datetime"].dt.hour < 1, "document.amount"].sum())

In [19]:
# print(list(receipts_raw[0]["document"].keys()))
# print(list(receipts_raw[0].keys()))

In [20]:
# print("Filas en df:", len(df))
# print("Fechas únicas en df:", df["datetime"].dt.date.unique())

In [21]:
# import requests
# import pandas as pd
# from datetime import datetime, timedelta

# # -----------------------------
# # CONFIGURACIÓN
# # -----------------------------

# ACCESS_TOKEN = token

# headers = {
#     "Authorization": f"Bearer {ACCESS_TOKEN}",
#     "X-Version": "1.0.0",
#     "Content-Type": "application/json"
# }

# DIA = "2026-06-22"


# # -----------------------------
# # 1. DESCARGA DE RECEIPTS (solo el día 22)
# # -----------------------------

# def get_receipts(datetime_from, datetime_to, limit=100):

#     url = "https://api.cassanova.com/documents/receipts"
#     all_receipts = []
#     start = 0

#     while True:

#         params = {
#             "start": start,
#             "limit": limit,
#             "datetimeFrom": datetime_from,
#             "datetimeTo": datetime_to,
#             "calculatedAmount": "true"
#         }

#         response = requests.get(url, headers=headers, params=params)
#         response.raise_for_status()
#         data = response.json()

#         receipts = data.get("receipts", [])

#         if len(receipts) == 0:
#             break

#         all_receipts.extend(receipts)
#         print(f"  Página {start//limit + 1}: {len(receipts)} tickets")

#         if len(receipts) < limit:
#             break

#         start += limit

#     print(f"Total tickets descargados: {len(all_receipts)}")

#     return pd.json_normalize(all_receipts), all_receipts


# print(f"--- Descargando receipts del {DIA} ---")
# df_22, receipts_raw_22 = get_receipts(DIA, DIA)

# df_22["datetime"] = pd.to_datetime(df_22["datetime"])

# print("\nFilas:", len(df_22))
# print("Fechas únicas:", df_22["datetime"].dt.date.unique())
# print("Suma document.amount:", df_22["document.amount"].sum())


# # -----------------------------
# # 2. SOLDBYTAX (mismo día, para comparar)
# # -----------------------------

# def get_sold_by_tax(datetime_from, datetime_to, ids_sales_point=None, limit=100):

#     url = "https://api.cassanova.com/reports/sold/taxes"
#     all_sold = []
#     start = 0
#     data = None

#     while True:
#         params = {
#             "start": start,
#             "limit": limit,
#             "datetimeFrom": f'"{datetime_from}"',
#             "datetimeTo": f'"{datetime_to}"',
#         }
#         if ids_sales_point:
#             params["idsSalesPoint"] = ids_sales_point

#         response = requests.get(url, headers=headers, params=params)
#         response.raise_for_status()
#         data = response.json()

#         all_sold.extend(data.get("sold", []))

#         if start + limit >= data["totalCount"]:
#             break
#         start += limit

#     return data, all_sold


# print(f"\n--- soldByTax del {DIA} ---")
# sold_data_22, sold_detalle_22 = get_sold_by_tax(DIA, DIA)
# print("totalSold:", sold_data_22["totalSold"])
# print("totalRefund:", sold_data_22["totalRefund"])
# print("totalQuantity:", sold_data_22["totalQuantity"])

# neto_22 = float(sold_data_22["totalSold"]) - float(sold_data_22["totalRefund"])
# print("Neto soldByTax:", neto_22)
# print("Suma receipts:", df_22["document.amount"].sum())
# print("Diferencia:", neto_22 - df_22["document.amount"].sum())


# # -----------------------------
# # 3. ANÁLISIS DE BORDES HORARIOS (solo con df_22)
# # -----------------------------

# print("\n--- Bordes horarios dentro del día 22 ---")
# print("Tickets antes de la 01:00:", (df_22["datetime"].dt.hour < 1).sum())
# print("Tickets después de las 23:00:", (df_22["datetime"].dt.hour >= 23).sum())


# # -----------------------------
# # 4. DIFERENCIA CABECERA vs LÍNEAS (con receipts_raw_22)
# # -----------------------------

# print("\n--- Diferencias cabecera vs suma de líneas ---")
# diffs = []
# for rec in receipts_raw_22:
#     doc = rec["document"]
#     header_amount = doc["amount"]
#     rows_sum = sum(row.get("amount", 0) for row in doc.get("rows", []))
#     if abs(header_amount - rows_sum) > 0.01:
#         diffs.append((rec["id"], header_amount, rows_sum))

# print(f"Tickets con diferencia: {len(diffs)}")
# for d in diffs[:10]:
#     print(d)

In [22]:
# # ¿Cuántos receipts del día 22 tienen orderSummary?
# con_order = df_22["document.orderSummary.id"].notna().sum()
# print("Receipts con orderSummary:", con_order, "de", len(df_22))

# # Compara amount del receipt vs amount del orderSummary para esos casos
# mask = df_22["document.orderSummary.id"].notna()
# comparacion = df_22.loc[mask, ["id", "document.amount", "document.orderSummary.amount"]]
# comparacion["diff"] = comparacion["document.amount"] - comparacion["document.orderSummary.amount"]
# print(comparacion)
# print("Suma total de diferencias:", comparacion["diff"].sum())

In [23]:
# import requests
# import pandas as pd

# def get_invoices(datetime_from, datetime_to, limit=100):

#     url = "https://api.cassanova.com/documents/invoices"
#     all_invoices = []
#     start = 0

#     while True:

#         params = {
#             "start": start,
#             "limit": limit,
#             "datetimeFrom": datetime_from,
#             "datetimeTo": datetime_to,
#             "calculatedAmount": "true"
#         }

#         response = requests.get(url, headers=headers, params=params)
#         response.raise_for_status()
#         data = response.json()

#         invoices = data.get("invoices", [])  # ajustar si la clave se llama distinto

#         if len(invoices) == 0:
#             break

#         all_invoices.extend(invoices)
#         print(f"  Página {start//limit + 1}: {len(invoices)} facturas")

#         if len(invoices) < limit:
#             break

#         start += limit

#     print(f"Total facturas descargadas: {len(all_invoices)}")

#     return pd.json_normalize(all_invoices), all_invoices


# DIA = "2026-06-22"

# print(f"--- Descargando invoices del {DIA} ---")
# df_inv_22, invoices_raw_22 = get_invoices(DIA, DIA)

# if not df_inv_22.empty:
#     print("\nColumnas:", df_inv_22.columns.tolist())
#     print("\nNº total invoices:", len(df_inv_22))

#     if "deferred" in df_inv_22.columns:
#         print("Nº deferred=True:", (df_inv_22["deferred"] == True).sum())
#         print("Nº deferred=False:", (df_inv_22["deferred"] == False).sum())

#     # Suma solo de las NO diferidas (para no duplicar receipts ya facturados)
#     if "document.amount" in df_inv_22.columns:
#         no_deferred = df_inv_22[df_inv_22["deferred"] != True] if "deferred" in df_inv_22.columns else df_inv_22
#         suma_invoices_no_deferred = no_deferred["document.amount"].sum()
#         print("\nSuma invoices NO diferidas:", suma_invoices_no_deferred)

#         total_combinado = df_22["document.amount"].sum() + suma_invoices_no_deferred
#         print("Receipts + invoices no diferidas:", total_combinado)
#         print("Diferencia vs soldByTax (2422.6):", 2422.6 - total_combinado)
# else:
#     print("No hay invoices ese día.")

In [24]:
# import requests
# import pandas as pd

# def get_documents(tipo, clave_respuesta, datetime_from, datetime_to, limit=100):
#     """
#     tipo: 'bills', 'creditnotes', 'ddts', 'orders', 'quotations'
#     clave_respuesta: nombre de la clave del JSON que contiene la lista (lo probamos)
#     """
#     url = f"https://api.cassanova.com/documents/{tipo}"
#     all_docs = []
#     start = 0

#     while True:
#         params = {
#             "start": start,
#             "limit": limit,
#             "datetimeFrom": datetime_from,
#             "datetimeTo": datetime_to,
#             "calculatedAmount": "true"
#         }

#         response = requests.get(url, headers=headers, params=params)
#         response.raise_for_status()
#         data = response.json()

#         # Si no sabemos la clave, la detectamos automáticamente
#         if clave_respuesta not in data:
#             posibles = [k for k, v in data.items() if isinstance(v, list)]
#             if not posibles:
#                 print(f"  [{tipo}] No se encontró ninguna lista en la respuesta: {list(data.keys())}")
#                 break
#             clave_respuesta = posibles[0]

#         docs = data.get(clave_respuesta, [])

#         if len(docs) == 0:
#             break

#         all_docs.extend(docs)

#         if len(docs) < limit:
#             break

#         start += limit

#     return pd.json_normalize(all_docs), all_docs, clave_respuesta


# DIA = "2026-06-22"
# tipos = {
#     "bills": "bills",
#     "creditnotes": "creditNotes",
#     "ddts": "ddts",
#     "orders": "orders",
#     "quotations": "quotations",
# }

# resultados = {}

# for tipo, clave in tipos.items():
#     print(f"\n--- {tipo} ---")
#     df_tipo, raw_tipo, clave_real = get_documents(tipo, clave, DIA, DIA)
#     print(f"Clave usada: {clave_real} | Nº documentos: {len(df_tipo)}")

#     if not df_tipo.empty:
#         print("Columnas:", df_tipo.columns.tolist())
#         if "document.amount" in df_tipo.columns:
#             suma = df_tipo["document.amount"].sum()
#             print(f"Suma document.amount: {suma}")
#         else:
#             print("(No hay columna document.amount, revisar columnas disponibles)")

#     resultados[tipo] = df_tipo

# print("\n\n=== RESUMEN ===")
# print("Receipts:", df_22["document.amount"].sum())
# for tipo, df_tipo in resultados.items():
#     if not df_tipo.empty and "document.amount" in df_tipo.columns:
#         print(f"{tipo}:", df_tipo["document.amount"].sum())
#     else:
#         print(f"{tipo}: 0 (vacío o sin columna amount)")

In [25]:
# import pandas as pd

# def get_sales_by_hour_completo(datetime_from, datetime_to, only_confirmed=True):

#     df_receipts, _ = get_receipts(datetime_from, datetime_to)
#     df_bills, _, _ = get_documents("bills", "bills", datetime_from, datetime_to)

#     # Añadimos una columna para distinguir el origen, por si luego quieres depurar
#     df_receipts["source"] = "receipt"
#     if not df_bills.empty:
#         df_bills["source"] = "bill"
#         df_todo = pd.concat([df_receipts, df_bills], ignore_index=True)
#     else:
#         df_todo = df_receipts

#     if only_confirmed and "document.confirmed" in df_todo.columns:
#         df_todo = df_todo[df_todo["document.confirmed"] == True]

#     df_todo["datetime"] = pd.to_datetime(df_todo["datetime"])
#     df_todo["date"] = df_todo["datetime"].dt.date
#     df_todo["hour"] = df_todo["datetime"].dt.hour

#     resumen = (
#         df_todo.groupby(["date", "hour"])
#         .agg(total_amount=("document.amount", "sum"), num_documentos=("id", "count"))
#         .reset_index()
#         .sort_values(["date", "hour"])
#     )

#     return resumen, df_todo


# resumen_22, df_todo_22 = get_sales_by_hour_completo("2026-06-22", "2026-06-22")
# print(resumen_22)
# print("\nTotal día:", df_todo_22["document.amount"].sum())

In [26]:
# import requests
# import pandas as pd
# from datetime import datetime, timedelta

# # -----------------------------
# # CONFIGURACIÓN
# # -----------------------------

# ACCESS_TOKEN = token

# headers = {
#     "Authorization": f"Bearer {ACCESS_TOKEN}",
#     "X-Version": "1.0.0",
#     "Content-Type": "application/json"
# }


# # -----------------------------
# # DESCARGA GENÉRICA POR CHUNKS DE MÁXIMO 3 DÍAS
# # -----------------------------

# def _descargar_por_chunks(url, clave_respuesta, datetime_from, datetime_to, limit=100):
#     """
#     Descarga documentos entre datetime_from y datetime_to, troceando
#     en ventanas de máximo 3 días (límite de la API) y paginando dentro
#     de cada ventana.
#     """

#     all_docs = []

#     current = datetime.strptime(datetime_from, "%Y-%m-%d")
#     end = datetime.strptime(datetime_to, "%Y-%m-%d")

#     while current <= end:

#         # Ventana máxima de 3 días (current, current+1, current+2)
#         chunk_end = min(current + timedelta(days=2), end)

#         start = 0

#         while True:

#             params = {
#                 "start": start,
#                 "limit": limit,
#                 "datetimeFrom": current.strftime("%Y-%m-%d"),
#                 "datetimeTo": chunk_end.strftime("%Y-%m-%d"),
#                 "calculatedAmount": "true"
#             }

#             response = requests.get(url, headers=headers, params=params)
#             response.raise_for_status()
#             data = response.json()

#             if clave_respuesta not in data:
#                 posibles = [k for k, v in data.items() if isinstance(v, list)]
#                 clave_real = posibles[0] if posibles else None
#             else:
#                 clave_real = clave_respuesta

#             docs = data.get(clave_real, []) if clave_real else []

#             if len(docs) == 0:
#                 break

#             all_docs.extend(docs)

#             if len(docs) < limit:
#                 break

#             start += limit

#         current = chunk_end + timedelta(days=1)

#     return all_docs


# def get_receipts(datetime_from, datetime_to, limit=100):
#     url = "https://api.cassanova.com/documents/receipts"
#     raw = _descargar_por_chunks(url, "receipts", datetime_from, datetime_to, limit)
#     print(f"Receipts descargados: {len(raw)}")
#     return pd.json_normalize(raw), raw


# def get_bills(datetime_from, datetime_to, limit=100):
#     url = "https://api.cassanova.com/documents/bills"
#     raw = _descargar_por_chunks(url, "bills", datetime_from, datetime_to, limit)
#     print(f"Bills descargados: {len(raw)}")
#     return pd.json_normalize(raw), raw


# # -----------------------------
# # VENTAS POR HORA (receipts + bills combinados)
# # -----------------------------

# def get_sales_by_hour(datetime_from, datetime_to, only_confirmed=True):

#     print(f"\n=== Descargando datos de {datetime_from} a {datetime_to} ===")

#     df_receipts, _ = get_receipts(datetime_from, datetime_to)
#     df_bills, _ = get_bills(datetime_from, datetime_to)

#     df_receipts["source"] = "receipt"

#     if not df_bills.empty:
#         df_bills["source"] = "bill"
#         df_todo = pd.concat([df_receipts, df_bills], ignore_index=True)
#     else:
#         df_todo = df_receipts

#     if df_todo.empty:
#         print("No se han descargado documentos.")
#         return pd.DataFrame(), df_todo

#     if only_confirmed and "document.confirmed" in df_todo.columns:
#         antes = len(df_todo)
#         df_todo = df_todo[df_todo["document.confirmed"] == True]
#         print(f"Filtrados no confirmados: {antes - len(df_todo)} documentos excluidos")

#     df_todo["datetime"] = pd.to_datetime(df_todo["datetime"])
#     df_todo["date"] = df_todo["datetime"].dt.date
#     df_todo["hour"] = df_todo["datetime"].dt.hour

#     resumen = (
#         df_todo.groupby(["date", "hour"])
#         .agg(
#             total_amount=("document.amount", "sum"),
#             num_documentos=("id", "count")
#         )
#         .reset_index()
#         .sort_values(["date", "hour"])
#     )

#     return resumen, df_todo


# # -----------------------------
# # USO: una semana completa
# # -----------------------------



In [27]:
# resumen_semana, df_semana = get_sales_by_hour("2026-06-22", "2026-06-28")

# print("\n--- Resumen por día y hora ---")
# print(resumen_semana)

# print("\n--- Total por día ---")
# print(df_semana.groupby("date")["document.amount"].sum())

# print("\n--- Total de la semana ---")
# print(df_semana["document.amount"].sum())